# Smart Nyuki — Wax Moth Detection with Synthetic Sticky-Board Images + YOLO11
**SIC Week 8 — computer vision.** We have no labelled photos of Smart Nyuki sticky boards yet, so this notebook:

1. **Generates synthetic sticky-board images** — yellow or white boards with grids, uneven lighting, blur and JPEG noise — with **wax moths** (labelled) and **distractors** (flies, gnats, debris — not labelled, so the model learns to ignore them). Bounding boxes come for free.
2. **Trains YOLO11n** (Ultralytics) on them.
3. Measures what the beekeeper cares about: **count error per board**.
4. Shows how to **fine-tune on real photos** (Roboflow export or your own) — the step that makes it actually work in the field.

> ⚠️ Synthetic moths are drawn shapes. A model trained only on them will **not** be reliable on real photos. Use synthetic data to test the pipeline and to pre-train; real labelled photos decide the final accuracy.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
import os, io, random, glob, shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
SEED = 42
random.seed(SEED); np.random.seed(SEED)
IMG = 640
ROOT = '/content/waxmoth_synth' if os.path.isdir('/content') else 'waxmoth_synth'

## 1. Synthetic sticky-board generator
Adult wax moths rest with wings folded like a tent, so from above they look like an elongated grey-brown teardrop (roughly 10–20 mm). On a phone photo of a board that is about 25–70 px at 640 px.

In [ ]:
def board():
    if random.random() < 0.7:   # yellow sticky trap
        base = np.array([225, 205, 50]) + np.random.randint(-20, 20, 3)
    else:                       # white board
        base = np.array([235, 235, 225]) + np.random.randint(-15, 10, 3)
    img = Image.new('RGB', (IMG, IMG), tuple(int(v) for v in base))
    d = ImageDraw.Draw(img)
    step = random.randint(40, 90); line = tuple(int(v * 0.82) for v in base)
    off = random.randint(0, step)
    for x in range(off, IMG, step): d.line([(x, 0), (x, IMG)], fill=line, width=random.choice([1, 2]))
    for y in range(off, IMG, step): d.line([(0, y), (IMG, y)], fill=line, width=random.choice([1, 2]))
    # uneven lighting
    a = np.array(img).astype(np.float32)
    gx, gy = np.meshgrid(np.linspace(0, 1, IMG), np.linspace(0, 1, IMG))
    ang = random.uniform(0, 2 * np.pi)
    grad = np.cos(ang) * gx + np.sin(ang) * gy
    grad = 0.75 + 0.3 * (grad - grad.min()) / (np.ptp(grad) + 1e-9)
    a *= grad[..., None]
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def moth_patch(L):
    """Draw one moth pointing right on a transparent patch."""
    S = int(L * 1.6); P = Image.new('RGBA', (S, S), (0, 0, 0, 0)); d = ImageDraw.Draw(P)
    cx = cy = S / 2; W = L * random.uniform(0.32, 0.42)
    tone = random.randint(85, 140)
    body = (tone, int(tone * 0.88), int(tone * 0.7), 255)
    wing = [(cx - L / 2, cy), (cx - L * 0.25, cy - W / 2), (cx + L * 0.45, cy - W * 0.42),
            (cx + L / 2, cy), (cx + L * 0.45, cy + W * 0.42), (cx - L * 0.25, cy + W / 2)]
    d.polygon(wing, fill=body)
    dark = tuple(int(c * 0.6) for c in body[:3]) + (255,)
    d.line([(cx - L * 0.4, cy), (cx + L * 0.48, cy)], fill=dark, width=max(1, int(L / 25)))
    for _ in range(random.randint(4, 12)):                      # wing speckles
        x = random.uniform(cx - L * 0.3, cx + L * 0.4); y = random.uniform(cy - W * 0.3, cy + W * 0.3); r = L / 30
        d.ellipse([x - r, y - r, x + r, y + r], fill=dark)
    hr = L * 0.08
    d.ellipse([cx - L / 2 - hr, cy - hr, cx - L / 2 + hr, cy + hr], fill=dark)   # head
    for s in (-1, 1):                                            # antennae
        d.line([(cx - L / 2, cy), (cx - L / 2 - L * 0.2, cy + s * L * 0.15)], fill=dark, width=1)
    return P

def fly_patch(L):
    S = int(L * 2); P = Image.new('RGBA', (S, S), (0, 0, 0, 0)); d = ImageDraw.Draw(P); c = S / 2
    for s in (-1, 1):
        d.ellipse([c - L * 0.1, c + s * L * 0.15 - L * 0.35, c + L * 0.7, c + s * L * 0.15 + L * 0.35] if s > 0 else
                  [c - L * 0.1, c - L * 0.5, c + L * 0.7, c - L * 0.0], fill=(200, 200, 210, 120))
    d.ellipse([c - L * 0.5, c - L * 0.22, c + L * 0.3, c + L * 0.22], fill=(30, 30, 30, 255))
    return P

def debris_patch(L):
    S = int(L * 1.5); P = Image.new('RGBA', (S, S), (0, 0, 0, 0)); d = ImageDraw.Draw(P)
    pts = [(S / 2 + np.cos(t) * L / 2 * random.uniform(0.4, 1), S / 2 + np.sin(t) * L / 2 * random.uniform(0.4, 1))
           for t in np.linspace(0, 2 * np.pi, random.randint(5, 9), endpoint=False)]
    col = random.choice([(110, 80, 40), (60, 50, 40), (150, 120, 70)])
    d.polygon(pts, fill=col + (255,)); return P

def place(img, patch, taken, max_iou=0.05):
    patch = patch.rotate(random.uniform(0, 360), resample=Image.BICUBIC, expand=True)
    bb = patch.getbbox()
    if bb is None: return None
    patch = patch.crop(bb); w, h = patch.size
    for _ in range(30):
        x, y = random.randint(0, IMG - w), random.randint(0, IMG - h)
        box = (x, y, x + w, y + h)
        if all(iou(box, t) <= max_iou for t in taken):
            img.paste(patch, (x, y), patch); taken.append(box); return box
    return None

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u else 0

def make_image():
    img = board(); taken, moths = [], []
    for _ in range(min(np.random.poisson(random.choice([2, 6, 12])), 30)):
        b = place(img, moth_patch(random.uniform(25, 70)), taken)
        if b: moths.append(b)
    for _ in range(random.randint(0, 15)):
        maker = random.choice([fly_patch, fly_patch, debris_patch])
        place(img, maker(random.uniform(8, 30)), taken)
    if random.random() < 0.6: img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.3)))
    a = np.array(img).astype(np.float32) + np.random.normal(0, random.uniform(2, 8), (IMG, IMG, 3))
    img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    buf = io.BytesIO(); img.save(buf, 'JPEG', quality=random.randint(55, 95)); buf.seek(0)
    return Image.open(buf).convert('RGB'), moths

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax in axes.ravel():
    im, boxes = make_image(); ax.imshow(im); ax.set_title(f'{len(boxes)} moths'); ax.axis('off')
    for b in boxes: ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec='red', lw=1.5))
plt.tight_layout(); plt.show()

## 2. Write the dataset in YOLO format
`images/{train,val,test}` and `labels/{train,val,test}` — one `.txt` per image with `class cx cy w h` (all 0–1).

In [ ]:
N = {'train': 1200, 'val': 200, 'test': 200}
shutil.rmtree(ROOT, ignore_errors=True)
for split, n in N.items():
    os.makedirs(f'{ROOT}/images/{split}', exist_ok=True); os.makedirs(f'{ROOT}/labels/{split}', exist_ok=True)
    for i in range(n):
        im, boxes = make_image()
        im.save(f'{ROOT}/images/{split}/{split}_{i:05d}.jpg', quality=92)
        with open(f'{ROOT}/labels/{split}/{split}_{i:05d}.txt', 'w') as f:
            for x0, y0, x1, y1 in boxes:
                f.write(f'0 {(x0 + x1) / 2 / IMG:.6f} {(y0 + y1) / 2 / IMG:.6f} {(x1 - x0) / IMG:.6f} {(y1 - y0) / IMG:.6f}\n')
with open(f'{ROOT}/data.yaml', 'w') as f:
    f.write(f'path: {ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: wax_moth\n')
print(open(f'{ROOT}/data.yaml').read())

## 3. Train YOLO11n
Start from COCO-pretrained weights — transfer learning needs far fewer images than training from scratch. Synthetic boards are simple, so 40 epochs is plenty.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(data=f'{ROOT}/data.yaml', epochs=40, imgsz=IMG, batch=16, seed=SEED,
                      patience=10, project='runs', name='waxmoth_synth', exist_ok=True)
best = YOLO('runs/waxmoth_synth/weights/best.pt')
m = best.val(data=f'{ROOT}/data.yaml', split='test')
print(f'test mAP@0.5 = {m.box.map50:.3f}   mAP@0.5:0.95 = {m.box.map:.3f}')

## 4. Count accuracy — the number the beekeeper sees

In [ ]:
def count_error(model, split='test', conf=0.25):
    true, pred = [], []
    for p in sorted(glob.glob(f'{ROOT}/images/{split}/*.jpg')):
        lab = p.replace('/images/', '/labels/').replace('.jpg', '.txt')
        true.append(sum(1 for _ in open(lab)))
        pred.append(len(model.predict(p, conf=conf, verbose=False)[0].boxes))
    return np.array(true), np.array(pred)

t, p = count_error(best)
print(f'count MAE: {np.abs(t - p).mean():.2f} moths/board | exact: {(t == p).mean():.0%} | within ±1: {(np.abs(t - p) <= 1).mean():.0%}')
plt.figure(figsize=(5, 5)); plt.scatter(t, p, alpha=0.5); lim = [0, max(t.max(), p.max()) + 1]
plt.plot(lim, lim, 'k--'); plt.xlabel('true moths'); plt.ylabel('predicted moths'); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, path in zip(axes, random.sample(glob.glob(f'{ROOT}/images/test/*.jpg'), 3)):
    r = best.predict(path, conf=0.25, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.set_title(f'{len(r.boxes)} moths detected'); ax.axis('off')
plt.show()

## 5. Fine-tune on real photos (the step that matters)
**Option A — Roboflow Universe.** Pick a moth dataset (check its licence), click *Download → YOLOv8 format → show download code*, and paste the workspace / project / version below. Add your key as `ROBOFLOW_API_KEY` in Colab **Secrets**.

**Option B — your own photos.** Photograph Smart Nyuki sticky boards, label them in Roboflow or CVAT (one class, `wax_moth`), export in YOLO format.

Make sure the real dataset's class list maps to `wax_moth`; merge or drop other classes first.

In [ ]:
USE_REAL = False      # set True once the details below are filled in
if USE_REAL:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT').version(1).download('yolov8')
    real = YOLO('runs/waxmoth_synth/weights/best.pt')          # start from the synthetic model
    real.train(data=f'{ds.location}/data.yaml', epochs=60, imgsz=IMG, batch=16, seed=SEED,
               patience=15, lr0=0.002, project='runs', name='waxmoth_real', exist_ok=True)
    best = YOLO('runs/waxmoth_real/weights/best.pt')
    print(best.val(data=f'{ds.location}/data.yaml', split='test').box.map50)

## 6. Export for the Smart Nyuki backend
ONNX runs in the Python backend with `onnxruntime` (no PyTorch needed). `count_moths()` is the function the upload endpoint calls; compare the count to a threshold to raise the dashboard alert.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=IMG)
print('exported:', onnx_path)

ALERT_THRESHOLD = 10   # moths per board — set with local beekeepers

def count_moths(image_path, model=best, conf=0.25):
    r = model.predict(image_path, conf=conf, verbose=False)[0]
    n = len(r.boxes)
    Image.fromarray(r.plot()[..., ::-1]).save('annotated.jpg')
    return {'count': n, 'alert': n >= ALERT_THRESHOLD, 'annotated_image': 'annotated.jpg'}

print(count_moths(glob.glob(f'{ROOT}/images/test/*.jpg')[0]))
try:
    from google.colab import files
    files.download(onnx_path); files.download('runs/waxmoth_synth/weights/best.pt')
except ImportError:
    pass